In [3]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from scipy import stats

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from matplotlib.patches import Patch
from matplotlib.lines import Line2D


# ============================================================
# Configuration
# ============================================================

OUTPUT_STEM = Path("import_padding_tpr")

CONFIGS = [
    ("Standard", "std"),
    ("Third-party", "third"),
    ("Combined", "comb"),
]

CONFIDENCE = 0.95
FONT_SIZE = 13

Y_LOWER = 0.78
Y_UPPER = 1.005
Y_TICK_STEP = 0.05


def command_line_paths() -> list:
    """Return positional path arguments, or [] when running in a notebook."""
    try:
        get_ipython
        return []
    except NameError:
        pass

    if "ipykernel" in Path(sys.argv[0]).name:
        return []

    return [a for a in sys.argv[1:] if not a.startswith("-")]


cli_paths = command_line_paths()

if len(cli_paths) > 0:
    DATA_FOLDER = Path(cli_paths[0])
if len(cli_paths) > 1:
    OUTPUT_STEM = Path(cli_paths[1])


# ============================================================
# Load and aggregate
# ============================================================

runs = pd.read_csv("tpr_runs.csv", encoding="utf-8-sig")


def mean_ci(model: str, variant: str, input_type: str) -> tuple:
    values = (
        runs[
            (runs["model"] == model)
            & (runs["attack_variant"] == variant)
            & (runs["input_type"] == input_type)
        ]
        .sort_values("run")["tpr"]
        .to_numpy(dtype=float)
    )

    if len(values) < 2:
        raise ValueError(
            f"Need at least 2 runs for model={model!r}, variant={variant!r}, "
            f"input={input_type!r}; found {len(values)}."
        )

    half_width = stats.t.ppf(
        0.5 + CONFIDENCE / 2, len(values) - 1
    ) * values.std(ddof=1) / np.sqrt(len(values))

    return values.mean(), half_width, len(values)


labels = [label for label, _ in CONFIGS]
keys = [key for _, key in CONFIGS]

orig_padded, orig_padded_ci = [], []
at_padded, at_padded_ci = [], []
at_unmod, at_unmod_ci = [], []
run_counts = set()

for key in keys:
    mean, half, count = mean_ci("original", key, "padded")
    orig_padded.append(mean)
    orig_padded_ci.append(half)
    run_counts.add(count)

    mean, half, count = mean_ci("adversarial", key, "padded")
    at_padded.append(mean)
    at_padded_ci.append(half)
    run_counts.add(count)

    mean, half, count = mean_ci("adversarial", key, "unmodified")
    at_unmod.append(mean)
    at_unmod_ci.append(half)
    run_counts.add(count)

orig_padded = np.array(orig_padded)
orig_padded_ci = np.array(orig_padded_ci)
at_padded = np.array(at_padded)
at_padded_ci = np.array(at_padded_ci)
at_unmod = np.array(at_unmod)
at_unmod_ci = np.array(at_unmod_ci)

orig_unmod, orig_unmod_ci, baseline_runs = mean_ci(
    "original", "none", "unmodified"
)
run_counts.add(baseline_runs)

if len(run_counts) != 1:
    print(
        f"warning: unequal run counts across conditions: "
        f"{sorted(run_counts)}"
    )

runs_per_condition = max(run_counts)

print(f"Conditions aggregated from {'tpr_runs.csv'}")
print(f"Runs per condition: {runs_per_condition}")
print(f"Baseline (Original + Unmodified): "
      f"{orig_unmod:.5f} +/- {orig_unmod_ci:.5f}\n")

figure_values = pd.DataFrame(
    {
        "Configuration": labels,
        "Original + Padded": orig_padded,
        "Original + Padded CI": orig_padded_ci,
        "AT + Padded": at_padded,
        "AT + Padded CI": at_padded_ci,
        "AT + Unmodified": at_unmod,
        "AT + Unmodified CI": at_unmod_ci,
        "Original + Unmodified": orig_unmod,
        "Original + Unmodified CI": orig_unmod_ci,
        "Runs": runs_per_condition,
    }
).round(5)

figure_values.to_csv(
    "figure_import_padding_values.csv",
    index=False,
    encoding="utf-8-sig",
)

print(figure_values.to_string(index=False))


# ============================================================
# Figure
# ============================================================

plt.rcParams.update(
    {
        "font.size": FONT_SIZE,
        "axes.labelsize": FONT_SIZE,
        "xtick.labelsize": FONT_SIZE,
        "ytick.labelsize": FONT_SIZE,
        "legend.fontsize": FONT_SIZE,
        "font.family": "serif",
    }
)

fig, ax = plt.subplots(figsize=(5.4, 3.65))

group_gap = 1.55
bar_width = 0.25

centers = np.arange(len(labels)) * group_gap
orig_padded_pos = centers - bar_width
at_padded_pos = centers
at_unmod_pos = centers + bar_width

error_style = {
    "ecolor": "black",
    "elinewidth": 1.0,
    "capsize": 3,
    "capthick": 1.0,
}

ax.bar(
    orig_padded_pos, orig_padded, width=bar_width, yerr=orig_padded_ci,
    error_kw=error_style, facecolor="white", edgecolor="black",
    linewidth=1.0, hatch="///", zorder=3,
)
ax.bar(
    at_padded_pos, at_padded, width=bar_width, yerr=at_padded_ci,
    error_kw=error_style, facecolor="0.65", edgecolor="black",
    linewidth=1.0, zorder=3,
)
ax.bar(
    at_unmod_pos, at_unmod, width=bar_width, yerr=at_unmod_ci,
    error_kw=error_style, facecolor="white", edgecolor="black",
    linewidth=1.0, hatch="...", zorder=3,
)

ax.axhspan(
    orig_unmod - orig_unmod_ci, orig_unmod + orig_unmod_ci,
    facecolor="0.88", edgecolor="none", alpha=0.8, zorder=1,
)
ax.axhline(
    y=orig_unmod, color="black", linestyle="--", linewidth=1.3, zorder=4
)

ax.set_ylabel("Mean True Positive Rate", fontsize=FONT_SIZE)

lower_edge = min(
    (orig_padded - orig_padded_ci).min(),
    (at_padded - at_padded_ci).min(),
    (at_unmod - at_unmod_ci).min(),
)

lower_bound = Y_LOWER

if lower_edge < Y_LOWER + 0.005:
    lower_bound = np.floor((lower_edge - 0.02) / Y_TICK_STEP) * Y_TICK_STEP
    print(
        f"warning: data reaches {lower_edge:.5f}, below the preferred "
        f"y-limit {Y_LOWER}; axis extended to {lower_bound:.2f}"
    )

first_tick = np.ceil((lower_bound + 0.015) / Y_TICK_STEP) * Y_TICK_STEP

ax.set_ylim(lower_bound, Y_UPPER)
ax.set_yticks(np.arange(first_tick, 1.0 + Y_TICK_STEP / 2, Y_TICK_STEP))
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
ax.tick_params(axis="both", labelsize=FONT_SIZE)

d = 0.012
kwargs = dict(
    transform=ax.transAxes, color="black", clip_on=False, linewidth=1.0
)
ax.plot((-d, +d), (-d, +d), **kwargs)
ax.plot((-d, +d), (0.025 - d, 0.025 + d), **kwargs)

ax.grid(axis="y", linestyle="--", linewidth=0.6, alpha=0.30, zorder=0)
ax.set_axisbelow(True)

ax.set_xticks(centers)
ax.set_xticklabels(labels, fontsize=FONT_SIZE)
ax.set_xlabel(
    "Import-padding Configuration", fontsize=FONT_SIZE, labelpad=8
)

for i in range(len(centers) - 1):
    sep = (centers[i] + centers[i + 1]) / 2
    ax.axvline(
        x=sep, color="0.60", linestyle=":", linewidth=0.9,
        ymin=0.04, ymax=0.96, zorder=1,
    )

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.margins(x=0.08)

legend_handles = [
    Patch(
        facecolor="white", edgecolor="black", hatch="///", linewidth=1.0,
        label="Original + Padded",
    ),
    Patch(
        facecolor="0.65", edgecolor="black", linewidth=1.0,
        label="AT + Padded",
    ),
    Patch(
        facecolor="white", edgecolor="black", hatch="...", linewidth=1.0,
        label="AT + Unmodified",
    ),
    Line2D(
        [0], [0], color="black", linestyle="--", linewidth=1.3,
        label="Original + Unmodified",
    ),
]

legend = fig.legend(
    handles=legend_handles, loc="upper center",
    bbox_to_anchor=(0.54, 0.995), ncol=2, frameon=True, fancybox=False,
    framealpha=1.0, facecolor="white", edgecolor="black",
    fontsize=FONT_SIZE, handlelength=1.35, handletextpad=0.35,
    columnspacing=1.6, borderpad=0.40,
)
legend.get_frame().set_linewidth(0.8)

plt.subplots_adjust(left=0.16, right=0.985, bottom=0.19, top=0.78)

plt.savefig(f"{OUTPUT_STEM}.pdf", bbox_inches="tight")
plt.savefig(f"{OUTPUT_STEM}.png", dpi=600, bbox_inches="tight")

print(f"\nsaved: {OUTPUT_STEM}.pdf / {OUTPUT_STEM}.png")

Conditions aggregated from tpr_runs.csv
Runs per condition: 10
Baseline (Original + Unmodified): 0.98972 +/- 0.00211

Configuration  Original + Padded  Original + Padded CI  AT + Padded  AT + Padded CI  AT + Unmodified  AT + Unmodified CI  Original + Unmodified  Original + Unmodified CI  Runs
     Standard            0.83178               0.00705      0.95327         0.00546          0.98131             0.00315                0.98972                   0.00211    10
  Third-party            0.89720               0.00630      0.97196         0.00446          0.98224             0.00379                0.98972                   0.00211    10
     Combined            0.85981               0.00772      0.96262         0.00546          0.97944             0.00423                0.98972                   0.00211    10

saved: import_padding_tpr.pdf / import_padding_tpr.png
